# indah - Chatbot (streaming)

A live demo of [**indah**](https://github.com/leejianrong/indah) - a reactive Python UI framework for cloud notebooks (no Node, single port, streaming over SSE).

The full app is laid out below - read it, tweak it, and re-run. **Runtime -> Run all**, then use the app that appears inline in the last cell.

In [ ]:
# Install indah from PyPI (plus any demo extras).
!pip install -q "indah"

In [ ]:
# chatbot.py - the complete demo, inline (no clone, no download).
"""A streaming LLM chatbot in indah, runnable end to end in a Colab cell.

The example is deliberately in two layers, kept apart (ADR-0009):

- ``chat_stream`` is plain Python with no indah imports. It loads a small
  instruct model with transformers and yields the reply token by token. A real
  app drops its own model behind the same shape; on graduation the function lifts
  out of indah unchanged.
- the indah layer (``build_session``) wires that stream into a UI: a message box,
  a Send button, and a ``Chat`` of role bubbles whose in-flight reply streams into
  a live pending bubble (ADR-0016).

Generation runs on a background thread and is pulled token by token with
``asyncio.to_thread``, so the event loop is never blocked and the rest of the UI
stays live while the model generates - the same non-blocking guarantee the demo's
mock LLM shows (R3, ADR-0011).

Run it::

    python examples/chatbot.py --mock          # no model, no GPU: canned replies
    python examples/chatbot.py                  # Qwen2.5-0.5B-Instruct (downloads)
    python examples/chatbot.py --model unsloth/Llama-3.2-1B-Instruct

``--mock`` needs only indah, so the wiring is runnable and smoke-checkable without
transformers, torch, or a GPU. The real path needs the user's own deps
(``pip install transformers accelerate`` plus torch); they are never indah's, so
indah stays pure-Python with no Node at install or runtime.
"""

from __future__ import annotations

import argparse
import asyncio
from collections.abc import AsyncIterator

import indah
from indah import (
    Button,
    Chat,
    Column,
    Expander,
    Row,
    Select,
    Session,
    Signal,
    Text,
    TextInput,
)

# The empty-state prompt chips: clickable suggestions that both teach what indah is and
# give a first-time visitor something to send without thinking of a prompt.
DEFAULT_SUGGESTIONS: list[str] = [
    "Explain how indah streams tokens",
    "Write a haiku about Colab",
    "What is a reactive signal?",
]

DEFAULT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# A message is a plain ``{"role": ..., "content": ...}`` dict, the shape
# transformers' chat templates expect. Kept as plain data so the domain layer
# never depends on indah.
Message = dict[str, str]


# --- the model layer: plain Python, no indah imports (ADR-0009) --------------


def load_model(model_name: str = DEFAULT_MODEL):
    """Load a small instruct model + tokenizer. Plain transformers.

    ``device_map="auto"`` puts the model on the GPU when Colab has a T4 and on the
    CPU otherwise; both work for a sub-1B model. The first call downloads weights.
    """
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")
    return tokenizer, model


async def chat_stream(
    tokenizer, model, messages: list[Message], *, max_new_tokens: int = 1024
) -> AsyncIterator[str]:
    """Yield an assistant reply token by token for a chat ``messages`` list.

    ``max_new_tokens`` caps how long a single reply may be. It defaults high (1024)
    and is exposed through ``build_session``/``--max-new-tokens`` so a reply is not
    truncated mid-sentence (KAN-1401). This is the generation cap only, distinct
    from the model's context window -- the small instruct models here have a large
    context (32k+), so 1024 new tokens never hits it.

    transformers' ``TextIteratorStreamer`` runs ``model.generate`` on a background
    thread and exposes a *blocking* iterator of tokens. We pull each token off it
    with ``asyncio.to_thread`` so the asyncio event loop is free between tokens and
    indah's UI never freezes while the model generates. No indah imports: this is
    the portable, graduation-clean half of the app.
    """
    import threading

    from transformers import TextIteratorStreamer

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    def generate() -> None:
        try:
            model.generate(
                **inputs,
                streamer=streamer,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
            )
        finally:
            # Signal end-of-stream even if generate() raised, so the consumer
            # below is never left blocked forever on a dead generation thread.
            streamer.end()

    threading.Thread(target=generate, daemon=True).start()

    done = object()
    while True:
        token = await asyncio.to_thread(next, streamer, done)
        if token is done:
            break
        yield token


async def mock_chat_stream(
    tokenizer, model, messages: list[Message], *, max_new_tokens: int = 1024
) -> AsyncIterator[str]:
    """A model-free stand-in with the same shape as ``chat_stream`` (for --mock).

    Lets the whole app - wiring, streaming, transcript - run and be smoke-tested
    with no transformers/torch and no GPU. It honours ``max_new_tokens`` (one word
    ~ one token here) so the same cap that bounds a real reply is exercised.
    """
    last = messages[-1]["content"].strip() if messages else ""
    reply = (f"You said: '{last}'. " if last else "") + (
        "This is a mock reply, streamed token by token. Pass a real model to "
        "chat_stream() to swap me out - the wiring does not change."
    )
    for word in reply.split(" ")[:max_new_tokens]:
        await asyncio.sleep(0.05)
        yield word + " "


# --- the keyed path: a real reply from Google Gemini (bring your own key) -----


async def gemini_chat_stream(
    api_key: str,
    messages: list[Message],
    *,
    max_new_tokens: int = 1024,
    model: str = "gemini-2.0-flash",
) -> AsyncIterator[str]:
    """Stream a real reply from Google Gemini, token by token (bring your own key).

    Plain Python, no indah imports (ADR-0009): given a free Google AI Studio API key
    and a chat ``messages`` list, it yields the model's reply as it arrives. It hits
    the REST ``streamGenerateContent`` endpoint (server-sent events) over stdlib
    ``urllib`` on a background thread, pulled with ``asyncio.to_thread`` so the event
    loop never blocks. The key is used only for this request and never stored or
    logged. On any error it yields a short message instead of raising, so the chat
    stays usable (e.g. a bad key or exhausted quota).
    """
    import json
    import queue as _queue
    import threading
    import urllib.error
    import urllib.request

    endpoint = (
        f"https://generativelanguage.googleapis.com/v1beta/models/{model}"
        f":streamGenerateContent?alt=sse&key={api_key}"
    )
    contents = [
        {"role": "user" if m["role"] == "user" else "model", "parts": [{"text": m["content"]}]}
        for m in messages
        if m["role"] in ("user", "assistant")
    ]
    payload = json.dumps(
        {"contents": contents, "generationConfig": {"maxOutputTokens": max_new_tokens}}
    ).encode()

    q: _queue.Queue = _queue.Queue()
    done = object()

    def pump() -> None:
        try:
            req = urllib.request.Request(
                endpoint, data=payload, headers={"Content-Type": "application/json"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:  # noqa: S310 (https only)
                for raw in resp:
                    line = raw.decode("utf-8").strip()
                    if not line.startswith("data:"):
                        continue
                    chunk = line[len("data:") :].strip()
                    if not chunk:
                        continue
                    try:
                        text = json.loads(chunk)["candidates"][0]["content"]["parts"][0]["text"]
                    except (KeyError, IndexError, ValueError):
                        continue
                    if text:
                        q.put(text)
        except urllib.error.HTTPError as exc:
            q.put(f"\n[Gemini error {exc.code}: check the API key or quota.]")
        except (urllib.error.URLError, OSError) as exc:
            q.put(f"\n[Could not reach Gemini: {exc}]")
        finally:
            q.put(done)

    threading.Thread(target=pump, daemon=True).start()
    while True:
        item = await asyncio.to_thread(q.get)
        if item is done:
            break
        yield item


# --- the keyed path: a real reply from OpenRouter (bring your own key) --------

# OpenRouter model picks for the demo, as ``(id, label)`` pairs for the Select. The
# default is a **free** model so nobody has to pay; the others are cheap paid picks for
# users who have credits. NOTE: OpenRouter's free-model roster rotates, so treat these
# ``:free`` ids as a starting set and edit them as the catalogue changes.
DEFAULT_OPENROUTER_MODEL = "deepseek/deepseek-chat-v3.1:free"
OPENROUTER_MODELS: list[tuple[str, str]] = [
    ("deepseek/deepseek-chat-v3.1:free", "DeepSeek V3.1 (free)"),
    ("qwen/qwen-2.5-72b-instruct:free", "Qwen 2.5 72B (free)"),
    ("meta-llama/llama-3.3-70b-instruct:free", "Llama 3.3 70B (free)"),
    ("deepseek/deepseek-chat", "DeepSeek (paid, cheap)"),
    ("qwen/qwen-2.5-72b-instruct", "Qwen 2.5 72B (paid)"),
]


async def openrouter_chat_stream(
    api_key: str,
    messages: list[Message],
    *,
    model: str = DEFAULT_OPENROUTER_MODEL,
    max_new_tokens: int = 1024,
) -> AsyncIterator[str]:
    """Stream a real reply from OpenRouter, token by token (bring your own key).

    Plain Python, no indah imports (ADR-0009). OpenRouter is OpenAI-compatible: we POST
    to ``/chat/completions`` with ``stream: true`` and read the server-sent ``data:``
    lines over stdlib ``urllib`` on a background thread, pulled with ``asyncio.to_thread``
    so the event loop never blocks. ``model`` picks which OpenRouter model answers; the
    default is a free one. The key is used only for this request and never stored or
    logged. On any error it yields a short message instead of raising, so the chat stays
    usable (e.g. a bad key, an unavailable model, or exhausted quota).
    """
    import json
    import queue as _queue
    import threading
    import urllib.error
    import urllib.request

    endpoint = "https://openrouter.ai/api/v1/chat/completions"
    payload = json.dumps(
        {
            "model": model,
            "stream": True,
            "max_tokens": max_new_tokens,
            "messages": [
                {"role": m["role"], "content": m["content"]}
                for m in messages
                if m["role"] in ("user", "assistant", "system")
            ],
        }
    ).encode()

    q: _queue.Queue = _queue.Queue()
    done = object()

    def pump() -> None:
        try:
            req = urllib.request.Request(
                endpoint,
                data=payload,
                headers={
                    "Content-Type": "application/json",
                    "Authorization": f"Bearer {api_key}",
                },
            )
            with urllib.request.urlopen(req, timeout=60) as resp:  # noqa: S310 (https only)
                for raw in resp:
                    line = raw.decode("utf-8").strip()
                    if not line.startswith("data:"):
                        continue
                    chunk = line[len("data:") :].strip()
                    if not chunk or chunk == "[DONE]":
                        continue
                    try:
                        text = json.loads(chunk)["choices"][0]["delta"].get("content") or ""
                    except (KeyError, IndexError, ValueError):
                        continue
                    if text:
                        q.put(text)
        except urllib.error.HTTPError as exc:
            q.put(_openrouter_error_message(exc))
        except (urllib.error.URLError, OSError) as exc:
            q.put(f"\n[Could not reach OpenRouter: {exc}]")
        finally:
            q.put(done)

    threading.Thread(target=pump, daemon=True).start()
    while True:
        item = await asyncio.to_thread(q.get)
        if item is done:
            break
        yield item


def _openrouter_error_message(exc) -> str:
    """Turn an OpenRouter HTTP error into a chat message that actually explains it.

    ``exc.code`` alone hides the real cause -- OpenRouter puts it in the JSON body
    (``{"error": {"message": ...}}``), so we read and surface that. Two failure modes
    are common enough with **free** (``:free``) models specifically to call out by
    name, since they leave paid models unaffected and are otherwise a confusing
    "free doesn't work, paid does" symptom:

    - **404 "no endpoints match your data policy"** -- OpenRouter only routes to a
      free model if the account has opted in to prompt training/"free model
      publication" at https://openrouter.ai/settings/privacy. Paid models have no
      such requirement, so they keep working while every free model 404s.
    - **429 upstream rate limit** -- free models share a heavily-throttled pool; the
      backing provider (e.g. Chutes) rejects bursts. Transient; retrying shortly or
      picking a different model usually clears it.
    """
    import json

    try:
        body = exc.read().decode("utf-8", errors="replace")
    except Exception:  # noqa: BLE001 - reading the error body is best-effort
        body = ""
    detail = ""
    if body:
        try:
            detail = json.loads(body).get("error", {}).get("message", "") or body
        except (ValueError, AttributeError):
            detail = body

    if exc.code == 404 and "data policy" in detail.lower():
        return (
            f"\n[OpenRouter error 404: {detail} Free models need "
            "'Free model publication' enabled at openrouter.ai/settings/privacy -- "
            "paid models don't need it, which is why only paid replies were working.]"
        )
    if exc.code == 429:
        return (
            f"\n[OpenRouter error 429: {detail or 'rate-limited upstream'}. Free "
            "models share a throttled pool; wait a bit or try a different model.]"
        )
    return f"\n[OpenRouter error {exc.code}: {detail or 'check the API key or model.'}]"


# --- the indah layer: wire the stream into a UI ------------------------------

# Coalesce a few tokens per SSE frame. Each StreamText.feed() emits one frame, and
# today every frame carries ~8 KB of proxy-flush padding (ADR-0002), so batching a
# handful of tokens per frame cuts the wire overhead several-fold with no
# perceptible loss of the streaming feel. A framework-level coalescing/debounce fix
# is tracked as a follow-up (see docs/QUESTIONS.md); this is the userland
# mitigation until then.
COALESCE_TOKENS = 3


async def _coalesced(
    stream: AsyncIterator[str], every: int = COALESCE_TOKENS
) -> AsyncIterator[str]:
    """Group ``every`` tokens into one chunk (with a final partial flush)."""
    buffer: list[str] = []
    async for token in stream:
        buffer.append(token)
        if len(buffer) >= every:
            yield "".join(buffer)
            buffer = []
    if buffer:
        yield "".join(buffer)


def build_session(
    stream_fn,
    tokenizer=None,
    model=None,
    *,
    max_new_tokens: int = 1024,
    api_key: Signal[str] | None = None,
    keyed_stream_fn=None,
    model_choice: Signal[str] | None = None,
    models: list[tuple[str, str]] | None = None,
    suggestions: list[str] | None = None,
) -> Session:
    """Build the chat UI around a ``stream_fn(tokenizer, model, messages)``.

    The conversation is a ``Signal[list]`` of ``{"role","content"}`` messages
    rendered as ``Chat`` bubbles (ADR-0016); the in-flight reply streams token by
    token into a ``pending`` signal so it shows as a live, growing assistant bubble,
    then commits to the list when done. Growing the transcript is an ordinary prop
    change over the existing patch op - no dynamic-children protocol op needed.

    ``max_new_tokens`` is threaded into every stream call so replies are not capped at
    the low library default and cut off mid-sentence (KAN-1401).

    Bring your own key (optional): pass ``api_key`` (a ``Signal[str]``) plus
    ``keyed_stream_fn(api_key, messages, *, max_new_tokens)`` and the key + a model
    ``Select`` (when ``model_choice`` + ``models`` are given) live in a collapsed
    **Settings** ``Expander`` -- out of the way, not clutter above the chat. When the key
    field holds a key, a send routes to ``keyed_stream_fn`` (a real model, using the chosen
    ``model_choice``); left blank it falls back to ``stream_fn`` (the mock). The key lives
    only in this session's signal and is never logged (ADR-0010).

    The empty state is a row of clickable suggestion chips (``suggestions``) that seed and
    send a prompt -- no explainer text, and the live streaming bubble is the only status.
    """
    prompt: Signal[str] = Signal("")
    busy: Signal[bool] = Signal(False)
    messages: Signal[list] = Signal([])
    pending: Signal[str] = Signal("")

    async def on_send() -> None:
        question = prompt.value.strip()
        if not question or busy.value:
            return
        busy.set(True)
        prompt.set("")  # clear the box for the next message
        history: list[Message] = messages.value + [{"role": "user", "content": question}]
        messages.set(history)
        pending.set("")

        # Route to the real (keyed) model when a key is present, else the mock. When a
        # model choice is exposed (OpenRouter), pass it through.
        key = api_key.value.strip() if api_key is not None else ""
        if key and keyed_stream_fn is not None:
            extra = {"model": model_choice.value} if model_choice is not None else {}
            stream = keyed_stream_fn(key, list(history), max_new_tokens=max_new_tokens, **extra)
        else:
            stream = stream_fn(tokenizer, model, list(history), max_new_tokens=max_new_tokens)

        parts: list[str] = []
        async for chunk in _coalesced(stream):
            pending.set(pending.value + chunk)
            parts.append(chunk)

        messages.set(messages.value + [{"role": "assistant", "content": "".join(parts)}])
        pending.set("")
        busy.set(False)

    async def on_chip(text: str) -> None:
        """Seed the box with a suggestion and send it (empty-state chips)."""
        if busy.value:
            return
        prompt.set(text)
        await on_send()

    # Settings (key + model choice) hide in a collapsed Expander, so the chat leads the
    # page instead of an API-key box and explainer text.
    children: list = []
    if api_key is not None:
        settings: list = []
        if model_choice is not None and models:
            settings.append(Select(model_choice, options=models, label="Model (OpenRouter)"))
        settings.append(
            TextInput(
                api_key,
                password=True,
                label="OpenRouter API key",
                placeholder="Paste a free key for a real reply; blank streams a mock",
            )
        )
        settings.append(
            Text(
                "The default model is free -- no cost. Get a free key at openrouter.ai. "
                "Your key is used only for this session and is never stored. Free "
                "models need 'Free model publication' enabled at "
                "openrouter.ai/settings/privacy, or replies will fail; the paid "
                "picks above have no such requirement."
            )
        )
        children.append(Expander(settings, label="Settings", open=False))

    children.append(Chat(messages, pending=pending, label="Conversation"))

    # Empty-state suggestion chips: click one to seed and send it (ghost buttons). They
    # replace the old "Ask me something." status string and teach what indah is.
    chips = [
        Button(text, on_click=(lambda t=text: on_chip(t)), variant="ghost")
        for text in (suggestions or DEFAULT_SUGGESTIONS)
    ]
    children.append(Row(chips, gap="0.5rem"))

    # Composer: the message box and Send, side by side. The live streaming bubble is the
    # only status -- no status line.
    children.append(
        Row(
            [
                TextInput(prompt, placeholder="Message", on_submit=on_send),
                Button("Send", on_click=on_send),
            ],
            gap="0.5rem",
            align="end",
        )
    )
    return Session(Column(children=children))


def main() -> None:
    parser = argparse.ArgumentParser(description="An indah streaming chatbot.")
    parser.add_argument(
        "--mock",
        action="store_true",
        help="canned replies with no model/GPU (needs only indah)",
    )
    parser.add_argument("--model", default=DEFAULT_MODEL, help="a Hugging Face model id")
    parser.add_argument(
        "--max-new-tokens",
        type=int,
        default=1024,
        help="cap on a single reply's length (raise if replies still cut off)",
    )
    args = parser.parse_args()

    if args.mock:
        session = build_session(mock_chat_stream, max_new_tokens=args.max_new_tokens)
    else:
        print(f"Loading {args.model} (the first run downloads weights)...", flush=True)
        tokenizer, model = load_model(args.model)
        session = build_session(chat_stream, tokenizer, model, max_new_tokens=args.max_new_tokens)

    indah.launch(indah.create_app(session=session))


# Module-level ASGI app for hosting (HF Spaces / uvicorn, ADR-0023): the mock
# chatbot, so a hosted demo needs no model weights or GPU. `python examples/chatbot.py`
# (main) still runs the real model by default; pass --mock for the same as here.
# The hosted demo: mock by default, but bring your own OpenRouter key for a real reply,
# choosing a model (a free one by default). A fresh api_key + model signal per session
# (never shared, never logged).
app = indah.create_app(
    session_factory=lambda: build_session(
        mock_chat_stream,
        api_key=Signal(""),
        keyed_stream_fn=openrouter_chat_stream,
        model_choice=Signal(DEFAULT_OPENROUTER_MODEL),
        models=OPENROUTER_MODELS,
    )
)

In [ ]:
# Launch it - in Colab this embeds the app inline in the cell output.
import indah
indah.launch(app)